
To access all project tables from the fund bronze data and create the Projects master table through UNION

In [0]:
all_bronze_tables = spark.sql("SHOW TABLES IN workspace.bronze_fund_data").toPandas()
all_bronze_tables

In [0]:
project_tables = [table for table in all_bronze_tables['tableName'] if table.endswith("projects")]

Read and create the master tables

In [0]:
bronze_path = "workspace.bronze_fund_data"

table_to_schema = spark.table(f"{bronze_path}.{project_tables[0]}")
master_project_df = spark.createDataFrame([],table_to_schema.schema)
# print(master_project_df.columns)

In [0]:
# main code to loop through the table and create the master project df

for t in range(len(project_tables)-6):
    print(f"adding {project_tables[t]}")
    table = spark.table(f"{bronze_path}.{project_tables[t]}")
    master_project_df = master_project_df.unionByName(table, allowMissingColumns=True)
    
    print(f"added {t} with {table.count()} records and {len(table.columns)} columns")
                                                
print(f"complete table created with {master_project_df.count()} records and {len(master_project_df.columns)} columns")



In [0]:
# display(master_project_df.groupBy('fund_id').count())
master_project_df.columns

Add Entity Code Descriptions

In [0]:
# create a temporary view of master_project_df for join

master_project_df.createOrReplaceTempView("master_project_view")

master_proj_w_entity_code = spark.sql("""
                                    SELECT mpd.*, ecd.Entity_Code_Desc
                                    FROM master_project_view AS mpd
                                    LEFT JOIN workspace.code_desc_project_master.entity_code_desc AS ecd
                                    ON mpd.Entity_Code = ecd.Entity_Code
                                    """)
display(master_proj_w_entity_code)


In [0]:
master_proj_w_entity_code.write.mode("overwrite").saveAsTable("workspace.code_desc_project_master.all_fund_projects")